# 02 · Preprocesamiento y Feature Engineering Temporal

**Objetivo (PDF sección 3.2):** generar variables temporales (ventanas, rezagos, agregaciones por periodo) y definir el split train/val/test respetando estrictamente el orden cronológico.

Decisiones de esta etapa, basadas en los hallazgos del notebook 01 (EDA temporal).

In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd

from data import load_transaction_data
from features import add_time_features, add_rolling_user_features, chronological_split

df = load_transaction_data()
df = add_time_features(df)
print(df.shape)

(590540, 437)


## 1. Estrategia de valores faltantes

El notebook 01 mostró que solo 24 de 438 columnas no tienen nulos, y que las columnas de `identity` (`id_01`-`id_38`, `DeviceType`, `DeviceInfo`) tienen cobertura muy inestable en el tiempo (16%-55%). Decisión:

- **Columnas de `identity`:** se descartan individualmente y se reemplazan por un único flag `has_identity` (¿la transacción tenía datos de identidad?) — mantener 41 columnas con >90% de nulos inestables en el tiempo no aporta señal confiable y complica el pipeline sin necesidad.
- **Numéricas restantes** (`C1-C14`, `D1-D15`, `V1-V339`, `dist1`, `dist2`): se imputan con **-999** (constante fuera de rango, no una estadística aprendida — no depende del split, así que no hay riesgo de fuga de información).
- **Categóricas** (`ProductCD`, `card4`, `card6`, `P_emaildomain`, `R_emaildomain`, `M1`-`M9`): los nulos se tratan como categoría propia ("missing") y se codifican a enteros (label encoding) — el mapeo de categoría→código es un vocabulario fijo, no una estadística del target, por lo que tampoco hay fuga al calcularlo antes del split.

In [1]:
identity_cols = [c for c in df.columns if c.startswith('id_') or c in ('DeviceType', 'DeviceInfo')]
df['has_identity'] = df['id_01'].notna().astype(int)
df = df.drop(columns=identity_cols)
print('columnas de identity descartadas:', len(identity_cols))
print('shape tras descartar identity:', df.shape)

categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('columnas categoricas:', categorical_cols)

for col in categorical_cols:
    df[col] = df[col].fillna('missing').astype('category').cat.codes

numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in ('TransactionID', 'isFraud')]
n_before = df[numeric_cols].isna().sum().sum()
df[numeric_cols] = df[numeric_cols].fillna(-999)
print('valores numericos nulos imputados con -999:', int(n_before))
print('nulos restantes en todo el dataframe:', int(df.isna().sum().sum()))

columnas de identity descartadas: 40
shape tras descartar identity: (590540, 398)
columnas categoricas: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
valores numericos nulos imputados con -999: 92362478
nulos restantes en todo el dataframe: 0


**Resultado:** se descartaron las columnas individuales de `identity`, quedando `has_identity` como señal resumen; el dataframe queda **sin nulos**, con las categóricas codificadas a enteros.

## 2. Tratamiento de outliers en `TransactionAmt`

El notebook 01 encontró un par de transacciones atípicas de $31,937.39. Se winsoriza `TransactionAmt` al percentil 99.5 (evita que un puñado de outliers domine el entrenamiento de modelos sensibles a escala como Regresión Logística) y se agrega `log_amt` (`log1p(TransactionAmt)`) como feature adicional, útil para modelos lineales dado que el monto tiene una distribución muy sesgada.

In [1]:
p995 = df['TransactionAmt'].quantile(0.995)
print('percentil 99.5 de TransactionAmt:', round(p995, 2))
print('transacciones por encima del percentil 99.5:', (df['TransactionAmt'] > p995).sum())

df['TransactionAmt'] = df['TransactionAmt'].clip(upper=p995)
df['log_amt'] = np.log1p(df['TransactionAmt'])
print(df[['TransactionAmt', 'log_amt']].describe())

percentil 99.5 de TransactionAmt: 1656.11
transacciones por encima del percentil 99.5: 2952
       TransactionAmt        log_amt
count   590540.000000  590540.000000
mean       131.381798       4.381313
std        196.398197       0.931396
min          0.251000       0.223943
25%         43.321000       3.791459
50%         68.769000       4.245190
75%        125.000000       4.836282
max       1656.110000       7.412830


## 3. Variables agregadas por ventana temporal

Se agrega, por `card1` (proxy de tarjeta/usuario), el conteo y monto promedio de transacciones en las **24 horas previas** — variable rezagada/agregada por periodo que pide el PDF (sección 3.2), y que solo mira hacia atrás en el tiempo (no usa información futura, por lo que es segura de calcular antes del split).

In [1]:
df = add_rolling_user_features(df, user_key='card1', window_seconds=86400)
new_cols = [c for c in df.columns if c.startswith('card1_')]
print('nuevas columnas:', new_cols)
print(df[new_cols].describe())

nuevas columnas: ['card1_txn_count_86400s', 'card1_amt_mean_86400s']
       card1_txn_count_86400s  card1_amt_mean_86400s
count           590540.000000          590540.000000
mean                19.803627             131.424886
std                 45.610592             134.380204
min                  1.000000               0.251000
25%                  2.000000              59.000000
50%                  6.000000             100.306458
75%                 20.000000             150.815705
max                881.000000            1656.110000


## 4. Split cronológico estricto (train / val / test)

60% / 20% / 20% respetando el orden temporal (sin aleatorizar), tal como exige la restricción de "Consistencia Temporal Estricta" del checkpoint 1 (sección 3) y el PDF del proyecto (sección 3.2: "evitando supuestos de independencia y de distribución idéntica").

In [1]:
train_df, val_df, test_df = chronological_split(df, dt_col='TransactionDT', train_frac=0.6, val_frac=0.2)

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: shape={part.shape}, dias={part["day"].min()}-{part["day"].max()}, '
          f'tasa_fraude={part["isFraud"].mean():.4f}')

train: shape=(354324, 401), dias=0-100, tasa_fraude=0.0338
val: shape=(118108, 401), dias=100-140, tasa_fraude=0.0390
test: shape=(118108, 401), dias=140-181, tasa_fraude=0.0344


## 5. Guardado de los datasets procesados

Se guardan en `data/processed/` en formato Parquet (más eficiente que CSV para reutilizar en los notebooks 03-05, dado el tamaño del dataset).

In [1]:
import os
os.makedirs('../data/processed', exist_ok=True)

train_df.to_parquet('../data/processed/train.parquet', index=False)
val_df.to_parquet('../data/processed/val.parquet', index=False)
test_df.to_parquet('../data/processed/test.parquet', index=False)

for name in ['train', 'val', 'test']:
    size_mb = os.path.getsize(f'../data/processed/{name}.parquet') / 1e6
    print(f'{name}.parquet: {size_mb:.1f} MB')

train.parquet: 57.9 MB
val.parquet: 18.7 MB
test.parquet: 19.9 MB


## Conclusiones para el notebook 03

- El dataframe queda sin nulos, con categóricas codificadas y listo para los 4 modelos candidatos sin pasos adicionales de preprocesamiento.
- `has_identity`, `log_amt`, `hour_of_day`, `day_of_week` y las variables `card1_txn_count_86400s` / `card1_amt_mean_86400s` son las features nuevas respecto al dataset crudo.
- El split respeta estrictamente el orden temporal — el notebook 03 debe entrenar solo con `train.parquet`, seleccionar hiperparámetros con `val.parquet`, y reportar la métrica final una sola vez sobre `test.parquet`.
- La tasa de fraude por split (ver sección 4) debe revisarse al interpretar resultados: al no ser un split aleatorio, no está garantizado que sea idéntica entre train/val/test — eso es justamente lo que se espera en un problema no estacionario.